# 🧪 TEST QUERY KHÓ — LocateAnything-3B nhận diện được tới đâu?

Kiểm tra khả năng **open-vocab** trên các query KHÓ cho MỌI bài toán:
**xe màu đỏ/trắng, phân biệt loại xe (car/truck/bus), phụ kiện người (ba lô, áo đỏ),
sản phẩm (cà chua chín, thùng carton), và tiếng Việt**. Với mỗi query, đo `det/frame`
→ **✅ nhận diện được** (det>0) hay **❌ không** — kèm ảnh có box.

Đây là thứ YOLO KHÔNG làm được (YOLO chỉ biết 80 lớp COCO, không hiểu màu/mô tả).
Chạy lần lượt các cell (cần GPU). Cell cuối in BẢNG kết quả + ảnh.

In [ ]:
# Cell 1 — Cài thư viện (đúng bản notebook Kaggle dùng được)
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless -q
!pip install -q -U "transformers==4.57.1" "opencv-python-headless==4.11.0.86" \
    "Pillow==11.1.0" "decord==0.6.0" "lmdb==1.7.5" accelerate peft "supervision>=0.21" matplotlib
print("✅ Đã cài. Nếu Colab báo 'Restart runtime' → Restart rồi chạy tiếp từ Cell 2.")

In [ ]:
# Cell 2 — Imports + kiểm tra GPU
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM (đặt TRƯỚC torch)
import re, time, glob, shutil, subprocess, urllib.request
from dataclasses import dataclass
from typing import List, Tuple, Optional
import numpy as np, cv2, torch, supervision as sv, transformers
from PIL import Image
import matplotlib.pyplot as plt

print("transformers:", transformers.__version__, "(cần 4.57.1)")
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU         :", p.name, f"{p.total_memory/1024**3:.1f}GB")

In [ ]:
# Cell 3 — parse_boxes: đổi text model → bbox (nguyên từ notebook chạy được)
NORM_SCALE = 1000
_RE_BOX = re.compile(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>")
_RE_BRK = re.compile(r"\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*\]")
_RE_LOC = re.compile(r"<loc_(\d+)>")

@dataclass
class Detection:
    bbox: Tuple[int, int, int, int]
    class_name: str
    confidence: float

def _add(dets, x1, y1, x2, y2, w, h, cls, conf, scale):
    x1, x2 = int(x1/scale*w), int(x2/scale*w)
    y1, y2 = int(y1/scale*h), int(y2/scale*h)
    if x2 > x1 and y2 > y1:
        dets.append(Detection((x1, y1, x2, y2), cls, conf))

def parse_boxes(text, class_name, w, h, default_conf=0.85):
    dets = []
    for m in _RE_BOX.findall(text):
        x1, y1, x2, y2 = (int(v) for v in m)
        _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    if not dets:
        for m in _RE_BRK.findall(text):
            v = [float(x) for x in m]
            sc = NORM_SCALE if max(v) > 1.5 else 1
            _add(dets, v[0], v[1], v[2], v[3], w, h, class_name, default_conf, sc)
    if not dets:
        locs = _RE_LOC.findall(text)
        for i in range(0, len(locs)-3, 4):
            x1, y1, x2, y2 = (int(v) for v in locs[i:i+4])
            _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    return dets

In [ ]:
# Cell 4 — LocateAnythingDetector (NGUYÊN recipe chạy được: device_map auto + generate thường)
class LocateAnythingDetector:
    def __init__(self, model_dir, max_new_tokens=1024):
        self.model_dir = model_dir
        self.max_new_tokens = max_new_tokens
        self._loaded = False
        self.dtype = torch.float16          # T4 (Turing) không có bfloat16 kernel

    def load(self):
        from transformers import AutoTokenizer, AutoProcessor, AutoConfig, AutoModel
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir, trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained(self.model_dir, trust_remote_code=True)
        config = AutoConfig.from_pretrained(self.model_dir, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(
            self.model_dir, config=config, trust_remote_code=True,
            torch_dtype=self.dtype, device_map="auto", attn_implementation="sdpa")
        self.model.eval()
        self._loaded = True
        print(f"✅ Loaded in {time.time()-t0:.1f}s")
        if torch.cuda.is_available():
            print(f"   GPU Mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
        return self

    def _prep_input(self, v):
        if isinstance(v, np.ndarray):
            v = torch.from_numpy(v)
        if torch.is_tensor(v):
            if v.is_floating_point():
                return v.to(device=self.model.device, dtype=torch.float16)
            return v.to(self.model.device)
        return v

    def detect_pil(self, pil_image, prompt, max_new_tokens=None):
        if not self._loaded: self.load()
        if torch.cuda.is_available(): torch.cuda.empty_cache()   # dọn VRAM phân mảnh trước mỗi frame → đỡ OOM
        w, h = pil_image.size
        max_tok = max_new_tokens or self.max_new_tokens
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": f"Locate all instances of: {prompt}"}]}]
        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=text_prompt, images=[pil_image], return_tensors="pt")
        inputs = {k: self._prep_input(v) for k, v in inputs.items()}
        with torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=max_tok,
                                         do_sample=False, use_cache=True, tokenizer=self.tokenizer)
        if isinstance(output, (list, tuple)) and hasattr(output[0], "shape"):
            raw = self.tokenizer.decode(output[0], skip_special_tokens=True)
        elif hasattr(output, "shape"):
            raw = self.tokenizer.decode(output[0] if output.dim() > 1 else output, skip_special_tokens=True)
        else:
            raw = str(output)
        return parse_boxes(raw, prompt, w, h), raw

    def detect_frame(self, bgr_frame, prompt, max_new_tokens=None):
        pil = Image.fromarray(cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB))
        return self.detect_pil(pil, prompt, max_new_tokens)

In [ ]:
# Cell 5 — Tải video test (xe/người của supervision + dây chuyền trong repo) + QUERY KHÓ
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.chdir(WORK)
REPO = os.path.join(WORK, "VisionOS"); BR = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BR,
                    "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git", REPO], check=True)
VID = os.path.join(REPO, "VisionOS", "sample_videos")

def _dl(name):
    p = os.path.join(WORK, name)
    if not os.path.exists(p):
        urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/" + name, p)
    return p

VEH = _dl("vehicles-2.mp4")        # giao lộ nhiều loại xe (rõ nét)
PPL = _dl("people-walking.mp4")    # người đi bộ
TOM = os.path.join(VID, "tomatoes_sorting.mp4")
BOX = os.path.join(VID, "packages_belt.mp4")

# (nhãn nhóm, video, [query khó]) — MÀU + LOẠI + PHỤ KIỆN + TIẾNG VIỆT + PHỦ ĐỊNH
TESTS = [
 ("XE", VEH, ["car", "truck", "bus", "a red car", "a white car", "a black car",
              "a large truck", "xe tải"]),
 ("NGƯỜI", PPL, ["person", "a person wearing a backpack", "a person in a red shirt",
                 "a person in white", "a woman", "người đội mũ"]),
 ("CÀ CHUA", TOM, ["object", "tomato", "a red tomato", "a green tomato", "fruit", "cà chua"]),
 ("KIỆN HÀNG", BOX, ["object", "carton box", "package", "box"]),
]
RESOLUTION = (1024, 576)   # nhỏ để đỡ OOM vision (hạ (896,512) nếu vẫn OOM)
N_FRAMES = 4               # mỗi query test N frame (LA chậm → để nhỏ)
MAX_NEW_TOKENS = 1024      # cell nạp model cần biến này
for name, path, qs in TESTS:
    print(("OK  " if os.path.exists(path) else "MISSING  "), f"{name:12}", path, "|", len(qs), "query")

In [ ]:
# Cell 6 — Tải model + patch bfloat16->float16 cho T4 (nguyên notebook chạy được)
from huggingface_hub import snapshot_download
MODEL_ID = "nvidia/LocateAnything-3B"
print("📥 Downloading model... (lần đầu ~6GB)")
model_dir = snapshot_download(MODEL_ID)
mc = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
if os.path.exists(mc): shutil.rmtree(mc)
f = os.path.join(model_dir, "modeling_locateanything.py")
if os.path.exists(f):
    real_f = os.path.realpath(f); code_txt = open(real_f).read()
    old = "pixel_values = pixel_values.to(self.language_model.dtype)"
    if old in code_txt:
        open(real_f, "w").write(code_txt.replace(old, "pixel_values = pixel_values.to(torch.float16)  # T4 fix"))
        print("✅ Patched bfloat16 -> float16")
print(f"📁 {model_dir}\n✅ Ready!")

In [ ]:
# Cell 7 — Nạp mô hình MỘT LẦN (dọn GPU trước để tránh OOM khi chạy lại)
import gc
if "detector" in globals():
    del detector
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU trống {free/1e9:.1f}/{total/1e9:.1f} GB trước khi nạp")
    if free/1e9 < 9:
        raise SystemExit("⚠️ GPU còn <9GB (model cũ KẸT). Run ▸ Restart runtime rồi chạy lại từ Cell 2 (mỗi cell 1 lần).")
detector = LocateAnythingDetector(model_dir=model_dir, max_new_tokens=MAX_NEW_TOKENS)
detector.load()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU trống sau khi nạp: {free/1e9:.1f}/{total/1e9:.1f} GB (cần >~3GB để chạy vision).")

In [ ]:
# Cell 8 — TEST: model nhận diện được query KHÓ nào? (det/frame + ✅/❌ + ảnh)
import time, math

def _no_full_frame(dets, w, h):
    return [d for d in dets if (d.bbox[2]-d.bbox[0])*(d.bbox[3]-d.bbox[1]) <= 0.9*w*h]

rows, shots = [], []
w, h = RESOLUTION
for task, path, queries in TESTS:
    if not os.path.exists(path):
        print("bỏ", task, "(thiếu video)"); continue
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for f in (0.3, 0.45, 0.6, 0.75)[:N_FRAMES]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(total*f)); ok, fr = cap.read()
        if ok: frames.append(cv2.resize(fr, RESOLUTION))
    cap.release()
    print(f"\n===== {task} ({len(frames)} frame/query) =====")
    for q in queries:
        tot, bestn, best = 0, -1, None
        t0 = time.time()
        for fr in frames:
            dets, _ = detector.detect_frame(fr, q)
            dets = _no_full_frame(dets, w, h)
            tot += len(dets)
            if len(dets) > bestn:
                bestn = len(dets); img = fr.copy()
                for d in dets:
                    x1, y1, x2, y2 = d.bbox; cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                best = img
        dpf = tot / max(len(frames), 1)
        ok = "✅" if dpf > 0 else "❌"
        rows.append((task, q, round(dpf, 1), ok))
        shots.append((f"{task}: {q} → {round(dpf,1)}", best))
        print(f"  {ok} {q:34} det/frame={dpf:.1f}  ({time.time()-t0:.0f}s)")

print("\n" + "=" * 60)
print("BẢNG: LocateAnything nhận diện QUERY KHÓ tới đâu")
print("=" * 60)
print(f"{'Bài':12}{'Query khó':36}{'det/fr':>7}  nhận diện")
print("-" * 60)
for task, q, dpf, ok in rows:
    print(f"{task[:11]:12}{q[:35]:36}{dpf:>7}  {ok}")
n_ok = sum(1 for r in rows if r[3] == "✅")
print(f"\n→ Nhận diện được {n_ok}/{len(rows)} query khó. "
      "(det/fr=0 = model chưa bắt được mô tả đó — thường là phủ định/quá trừu tượng/tiếng Việt.)")

# ảnh: mỗi query 1 frame nhiều box nhất
k = len(shots); cols = 4; nr = max(1, math.ceil(k / cols))
fig, axes = plt.subplots(nr, cols, figsize=(20, 4 * nr))
axf = list(np.atleast_1d(axes).flat)
for ax, (title, img) in zip(axf, shots):
    if img is not None: ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title[:42], fontsize=8); ax.axis("off")
for ax in axf[k:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Đọc bảng
- **✅ (det/fr>0)** = model HIỂU và bắt được query đó (màu 'a red car', loại 'truck',
  phụ kiện 'a person wearing a backpack', 'carton box', 'tomato'…). Đây là điểm mạnh
  open-vocab mà YOLO không có.
- **❌ (det/fr=0)** = chưa bắt được — thường là **phủ định** ('a person NOT wearing…'),
  **quá trừu tượng**, hoặc **tiếng Việt** (LA hiểu tiếng Việt kém → dùng tiếng Anh).
- Muốn kỹ hơn: tăng `N_FRAMES` (Cell 5). OOM: hạ `RESOLUTION` xuống (896,512).
- Đây là bản test NHANH (vài frame/query). Muốn chạy đầy đủ trên video + đếm: dùng
  `run_scenarios.py --suite` (xem README/SERVICE).